# Model development

In [ ]:
import pandas as pd 

df = pd.read_csv("taxi_cleaned_training_data.csv")
df.head()

In [ ]:
df.info()
df.isna().sum()

## Sckit learn steps

Divide into X and y
- X = what the model can know
- y = what the model get's to predict

In [ ]:
X, y = df.drop(columns="Trip_Price", axis=1), df["Trip_Price"]
X.head(2)

In [ ]:
y.head(2)

## Train | test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.33, random_state=42
)

## Dummy Encoding
- Categorical features -> represent with 1 and 0
- Categorical feature: Time_of_Day

In [ ]:
X_train = pd.get_dummies(X_train, drop_first=True).astype(int)
X_test = pd.get_dummies(X_test, drop_first=True).astype(int)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

## Model Training 


### Scale dataset
- min-max also called normalization
- feature standardization

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaler.fit(X_train)

scaled_X_train = scaler.transform(X_train)
scaled_X_test = scaler.transform(X_test)

scaled_X_train.shape, scaled_X_test.shape

In [ ]:
scaled_X_train.min(), scaled_X_train.max()

In [ ]:
scaled_X_test.min(), scaled_X_test.max()

## Linear Regression model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

model = LinearRegression()
model.fit(scaled_X_train, y_train)

model.intercept_, model.coef_
y_pred = model.predict(scaled_X_test)

mae_lr = mean_absolute_error(y_test, y_pred)
mse_lr = mean_squared_error(y_test, y_pred)
rmse_lr = np.sqrt(mse_lr)

mae_lr, mse_lr, rmse_lr


## KNN model

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

model = KNeighborsRegressor(n_neighbors=5)
model.fit(scaled_X_train, y_train)

y_pred = model.predict(scaled_X_test)

mae_knn = mean_absolute_error(y_test, y_pred)
mse_knn = mean_squared_error(y_test, y_pred)
rmse_knn = np.sqrt(mse_knn)

mae_knn, mse_knn, rmse_knn

## Random Forest model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

clf = RandomForestRegressor()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred)
mse_rf = mean_squared_error(y_test,y_pred)
rmse_rf = np.sqrt(mse_rf)

mae_rf, mse_rf, rmse_rf

## Comparisson of all models

In [ ]:
results = {}

results["LinearRegression"] = {
    "MAE": mae_lr,
    "RMSE": rmse_lr
}

results["KNN"] = {
    "MAE": mae_knn,
    "RMSE": rmse_knn
}

results["RandomForest"] = {
    "MAE": mae_rf,
    "RMSE": rmse_rf
}

results

In [ ]:
import matplotlib.pyplot as plt
residuals = y_test - y_pred 

plt.scatter(y_test, residuals)
plt.axhline(0, color="red")
plt.xlabel("True Price")
plt.ylabel("Residual")
plt.show

## Conclussion

After testing serveral regression models according to the data science workflow, linear regression showed the best results according to both MAE and RMSE. This indicates that the relationship in the data are largely linear and the model was therefore selected for further use.

## Final traning for production

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler

X, y  = df.drop(columns="Trip_Price"), df["Trip_Price"]

X_encoded = pd.get_dummies(X, drop_first=True)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_encoded)

final_model = LinearRegression()
final_model.fit(X_scaled, y)

In [ ]:
import joblib

joblib.dump({"model": final_model, "scaler": scaler, "features":X_encoded}, "artifact.joblib")